# Collecteur Jetstream

Ce notebook contient les fonctions de collecte : connexion au flux, filtrage des posts dont la langue déclarée contient `fr`, stockage SQLite et reconnexion. Il ne lance pas de collecte et ne modifie aucune base lors de son exécution.

Pour configurer et lancer la collecte, ouvrir `01_collecte.ipynb`, qui charge ces cellules de définitions. Les tests utilisent exactement les mêmes fonctions.

## 1. Imports et adresse du flux

Le flux est limité aux événements de la collection `app.bsky.feed.post`. Le filtrage de la langue est effectué dans `process_event`.

In [ ]:
import asyncio
import json
import sqlite3
from pathlib import Path
from urllib.parse import urlencode

import websockets
from websockets.exceptions import WebSocketException

URL = "wss://jetstream2.us-east.bsky.network/subscribe?wantedCollections=app.bsky.feed.post"


## 2. Repérer les liens externes

On conserve un indicateur de lien : carte externe, carte imbriquée dans une citation, ou lien des annotations du texte. La présence de lien ne conditionne pas la collecte.

In [ ]:
def has_external_link(rec):
    embed = rec.get("embed", {})

    if embed.get("$type") == "app.bsky.embed.external":
        return True
    if embed.get("$type") == "app.bsky.embed.recordWithMedia":
        if embed.get("media", {}).get("$type") == "app.bsky.embed.external":
            return True

    for facet in rec.get("facets", []):
        for feature in facet.get("features", []):
            if feature.get("$type") == "app.bsky.richtext.facet#link":
                return True

    return False

## 3. Créer la table et enregistrer une publication

`uri` est une clé primaire. `INSERT OR REPLACE` met à jour une URI existante. La table conserve le texte original et les métadonnées ; aucun nettoyage ou dédoublonnage de texte n’est appliqué ici.

In [ ]:
def create_database(connection):
    connection.execute("""
        CREATE TABLE IF NOT EXISTS posts (
            uri TEXT PRIMARY KEY,
            did TEXT NOT NULL,
            created_at TEXT,
            text TEXT NOT NULL,
            langs TEXT,
            is_reply INTEGER NOT NULL,
            has_link INTEGER NOT NULL
        )
    """)
    connection.commit()


def save_post(connection, post):
    connection.execute("""
        INSERT OR REPLACE INTO posts
        (uri, did, created_at, text, langs, is_reply, has_link)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (
        post["uri"],
        post["did"],
        post["created_at"],
        post["text"],
        post["langs"],
        int(post["is_reply"]),
        int(post["has_link"]),
    ))

## 4. Traiter un événement

Les suppressions reçues retirent la publication de SQLite. Pour une création ou modification, le post doit déclarer exactement `fr` dans `langs`. Un post multilingue peut être conservé ; sans déclaration de langue, il est ignoré. Une mise à jour qui ne contient plus `fr` retire l’ancienne ligne. La fonction retourne 1 uniquement pour une nouvelle URI française enregistrée.

In [ ]:
def process_event(connection, ev):
    """Apply creates, updates and deletes. Return 1 only for a new French URI."""
    if ev.get("kind") != "commit":
        return 0
    c = ev.get("commit", {})
    if c.get("collection") != "app.bsky.feed.post":
        return 0
    if not ev.get("did") or not c.get("rkey"):
        return 0
    uri = f"at://{ev['did']}/{c['collection']}/{c['rkey']}"
    operation = c.get("operation")
    if operation == "delete":
        connection.execute("DELETE FROM posts WHERE uri = ?", (uri,))
        return 0
    if operation not in ("create", "update"):
        return 0
    rec = c.get("record") or {}
    langs = rec.get("langs") or []
    if "fr" not in langs:
        connection.execute("DELETE FROM posts WHERE uri = ?", (uri,))
        return 0
    exists = connection.execute("SELECT 1 FROM posts WHERE uri = ?", (uri,)).fetchone()
    save_post(connection, {
        "uri": uri, "did": ev["did"], "created_at": rec.get("createdAt"),
        "text": rec.get("text", ""), "langs": ",".join(langs),
        "is_reply": "reply" in rec, "has_link": has_external_link(rec),
    })
    return int(exists is None)

## 5. Boucle asynchrone de collecte

`collect_posts` ouvre SQLite et une connexion WebSocket, puis traite les événements dans des transactions. Les limites portent sur les nouvelles URI et, si demandé, la durée. Après une coupure, la reconnexion utilise le dernier curseur reçu avec un délai progressif. Le curseur reste en mémoire : il ne permet pas une reprise durable après fermeture du noyau. La connexion SQLite est fermée dans `finally`, y compris en cas d’interruption.

Cette cellule définit seulement la fonction. Le lancement avec `await` est dans `01_collecte.ipynb`.

In [ ]:
async def collect_posts(max_posts=1000, duration_minutes=None, database_path="posts.db", max_retries=10):
    n = 0
    connection = sqlite3.connect(database_path)
    create_database(connection)
    loop = asyncio.get_running_loop()
    deadline = None if duration_minutes is None else loop.time() + duration_minutes * 60
    cursor, failures = None, 0
    try:
        while not max_posts or n < max_posts:
            if deadline is not None and loop.time() >= deadline:
                break
            url = URL if cursor is None else URL + "&" + urlencode({"cursor": cursor})
            try:
                remaining = None if deadline is None else max(0, deadline - loop.time())
                async with asyncio.timeout(remaining):
                    async with websockets.connect(url, open_timeout=30) as ws:
                        while not max_posts or n < max_posts:
                            ev = json.loads(await ws.recv())
                            with connection:
                                added = process_event(connection, ev)
                            if isinstance(ev.get("time_us"), int):
                                cursor = max(cursor or 0, ev["time_us"])
                            failures = 0
                            n += added
                            if added and n % 100 == 0:
                                print(f"{n} nouveaux posts français collectés", flush=True)
            except TimeoutError:
                if deadline is not None and loop.time() >= deadline:
                    break
                failures += 1
            except (OSError, WebSocketException) as error:
                failures += 1
                print(f"Connexion interrompue : {error}", flush=True)
            if failures:
                if failures > max_retries:
                    raise RuntimeError("Nombre maximal de reconnexions dépassé.")
                delay = min(2 ** failures, 60)
                if deadline is not None:
                    delay = min(delay, max(0, deadline - loop.time()))
                print(f"Reconnexion dans {delay:.0f} s…", flush=True)
                await asyncio.sleep(delay)
    finally:
        connection.close()
        print(f"Collecte terminée : {n} nouveaux posts dans {Path(database_path).name}.", flush=True)